In [29]:
import os

from typing import BinaryIO

def find_chunk_boundaries(
    file: BinaryIO,
    desired_num_chunks: int,
    split_special_token: bytes,
) -> list[int]:
    """
    Chunk the file into parts that can be counted independently.
    May return fewer chunks if the boundaries end up overlapping.
    """
    assert isinstance(split_special_token, bytes), "Must represent special token as a bytestring"

    # Get total file size in bytes
    file.seek(0, os.SEEK_END)
    file_size = file.tell()
    file.seek(0)

    chunk_size = file_size // desired_num_chunks

    # Initial guesses for chunk boundary locations, uniformly spaced
    # Chunks start on previous index, don't include last index
    chunk_boundaries = [i * chunk_size for i in range(desired_num_chunks + 1)]
    chunk_boundaries[-1] = file_size

    mini_chunk_size = 4096  # Read ahead by 4k bytes at a time

    for bi in range(1, len(chunk_boundaries) - 1):
        initial_position = chunk_boundaries[bi]
        file.seek(initial_position)  # Start at boundary guess
        while True:
            mini_chunk = file.read(mini_chunk_size)  # Read a mini chunk

            # If EOF, this boundary should be at the end of the file
            if mini_chunk == b"":
                chunk_boundaries[bi] = file_size
                break

            # Find the special token in the mini chunk
            found_at = mini_chunk.find(split_special_token)
            if found_at != -1:
                chunk_boundaries[bi] = initial_position + found_at
                break
            initial_position += mini_chunk_size

    # Make sure all boundaries are unique, but might be fewer than desired_num_chunks
    print(f"File size {file_size} Chunk boundaries {chunk_boundaries}")
    return sorted(set(chunk_boundaries))

In [33]:
import time
from collections import Counter
import itertools
import regex as re
from concurrent.futures import ProcessPoolExecutor

infile = '../data/TinyStoriesV2-GPT4-valid.txt'
errorcheck = "strict"
#errorcheck = "ignore"

def get_pretoken_counts(special_tokens:list[str],filename:str,start_end:tuple[int,int])->dict[str,int]:
    PAT = r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""
    counts_by_pretoken:dict[str,int] = Counter({})
    (start,end)  = start_end
    with open(filename,'rb') as f:
        f.seek(start)
        chunk = f.read(end - start).decode("utf-8", errors=errorcheck)
        regex_pattern = "|".join(map(re.escape,special_tokens))
            # only taking the first 'document' for prototyping. when things are working, will generalize remaining code into a function and loop over elements
        file_content_docs = re.split(regex_pattern,chunk)
        for file_content in file_content_docs:
            #logger.debug(f"regex pattern is {regex_pattern} Filtered {file_content}")
            pretoken_iter = re.finditer(PAT, file_content)
            for pretoken in pretoken_iter:
                pretoken_str = pretoken.group(0)
                counts_by_pretoken[pretoken_str] = counts_by_pretoken.get(pretoken_str, 0) + 1

        return counts_by_pretoken

num_processes = 4
with open(infile,'rb') as f:
    boundaries = find_chunk_boundaries(f, num_processes, b"<|endoftext|>")
    special_tokens=["<|endoftext|>"]
    special_tokens =sorted(special_tokens,key=lambda k:len(k),reverse=True)
    # The following is a serial implementation, but you can parallelize this
    # by sending each start/end pair to a set of processes.
    start_end_list = [(start,end) for start, end in zip(boundaries[:-1], boundaries[1:])]
    print(f"Start end {start_end_list}")

    start_time = time.perf_counter()
    with ProcessPoolExecutor() as executor:
         # executor.map applies the function to each element in parallel
         sizes = list(executor.map(get_pretoken_counts, itertools.repeat(special_tokens),itertools.repeat(infile),start_end_list))
    parallel_list = sizes
    parallel_dict = Counter({})
    for e in parallel_list:
        parallel_dict+=e


    end_time = time.perf_counter()
    execution_time = end_time - start_time
    print(f"Parallel execution time is {execution_time}")
    # for size in sizes:
    #     print(f"Process returned counts {len(size)} with first entry {next(iter(size.items()))}\n\n\n")
    boundaries = find_chunk_boundaries(f, num_processes, b"<|endoftext|>")
    serial_list = []
    start_time = time.perf_counter()
    for (start, end) in zip(boundaries[:-1], boundaries[1:]):
        print(f"Serial running from {start} to {end}")
        f.seek(start)
        chunk = f.read(end - start).decode("utf-8", errors=errorcheck)
        serial_list.append(get_pretoken_counts(special_tokens,infile,(start,end)))
    end_time = time.perf_counter()
    serial_dict = Counter({})
    for e in serial_list:
        serial_dict+=e

    execution_time = end_time - start_time
    print(f"Serial execution time is {execution_time}")

    # whole file no chunking
    boundaries = find_chunk_boundaries(f, 1, b"<|endoftext|>")
    whole_list = []
    start_time = time.perf_counter()

    for (start, end) in zip(boundaries[:-1], boundaries[1:]):
        print(f"Whole running from {start} to {end}")
        f.seek(start)
        chunk = f.read(end - start).decode("utf-8", errors=errorcheck)
        whole_list.append(get_pretoken_counts(special_tokens,infile,(start,end)))
    end_time = time.perf_counter()
    whole_dict = Counter({})
    for e in whole_list:
        whole_dict+=e
    execution_time = end_time - start_time
    print(f"Single chunk execution time is {execution_time}")


    print(f"Lengths of lists parallel:{len(parallel_dict)} serial: {len(serial_dict)} whole: {len(whole_dict)}")

File size 22502601 Chunk boundaries [0, 5625758, 11252559, 16877372, 22502601]
Start end [(0, 5625758), (5625758, 11252559), (11252559, 16877372), (16877372, 22502601)]
Parallel execution time is 5.5877323410022655
File size 22502601 Chunk boundaries [0, 5625758, 11252559, 16877372, 22502601]
Serial running from 0 to 5625758
Serial running from 5625758 to 11252559
Serial running from 11252559 to 16877372
Serial running from 16877372 to 22502601
Serial execution time is 14.254572885001835
File size 22502601 Chunk boundaries [0, 22502601]
Whole running from 0 to 22502601
Single chunk execution time is 14.297022188999108
Lengths of lists parallel:13111 serial: 13111 whole: 13111


In [ ]:
assert(whole_dict==serial_dict)
assert(parallel_dict==serial_dict)

In [6]:
import regex as re
special_tokens=['<end>','gary']
regex_pattern = "|".join(map(re.escape, special_tokens))
regex_pattern

'<end>|gary'